In [ ]:
import urllib

import geopandas as gpd
import pandas as pd

from openplaces.api import get_admin, read_entities
from openplaces.geo.vector import points_from_coords
from openplaces.io import share
from openplaces.path import external_path, share_path
from openplaces.viz import show_building

In [ ]:
def create_street_view_link(row):
    # Combine the components into a single string
    full_address = ', '.join(row)

    # URL encode the string (e.g., spaces become '+')
    encoded_address = urllib.parse.quote_plus(full_address)

    # Construct the final URL
    base_url = 'https://www.google.com/maps/search/?api=1&query='
    return f'{base_url}{encoded_address}'

In [ ]:
BUILDINGS_CHEER_PATH = external_path(
    'US-NC', 'building-cheer-v0', filename='Inventory_v0_NC.parquet'
)

CHEER_ADMIN3_IDS = (
    'US-NC-BA US-NC-BT US-NC-BL US-NC-BS US-NC-CD US-NC-CE US-NC-CW US-NC-CM US-NC-CN '
    'US-NC-CU US-NC-CI US-NC-DE US-NC-DP US-NC-ED US-NC-FR US-NC-GT US-NC-GE US-NC-HL '
    'US-NC-HT US-NC-HD US-NC-HO US-NC-HE US-NC-JH US-NC-JN US-NC-LN US-NC-MR US-NC-NA '
    'US-NC-NE US-NC-NO US-NC-ON US-NC-PM US-NC-PU US-NC-PD US-NC-PQ US-NC-PI US-NC-RB '
    'US-NC-SP US-NC-SC US-NC-TY US-NC-WK US-NC-WR US-NC-WI US-NC-WY US-NC-WO'
).split(' ')

PARCEL_COLUMNS_TO_LINK = [
    'parcel_id_admin3',
    'address',
    'city',
    'year_built',
    'improvement_value',
    'land_value',
]

REPROCESS = True

# Create inventory
- Join CHEER buildings to unique parcels
- Identify duplicate parcels and multi-building parcels
- Save and upload to Drive

Eastern North Carolina (CHEER) runs in ~1 min + upload

In [ ]:
CATEGORICAL_COLUMNS = [
    'county',
    'admin3_id',
    'foundation',
    'construction',
    'roof_shape_1',
    'roof_shape_2',
    'roof_shape_3',
]
out_path = share_path('US-NC', 'building-cheer-v01', filename='Eastern')
if out_path.exists() and not REPROCESS:
    building_inventory = gpd.read_parquet(out_path)
    print('Read inventory')
else:
    building_inventory_list = []
    for admin3_id in CHEER_ADMIN3_IDS:
        print(admin3_id, end=', ')

        # Read CHEER inventory
        county_fips = get_admin(admin3_id)['admin3_id_admin1'].values[0]
        buildings_cheer = gpd.read_parquet(
            BUILDINGS_CHEER_PATH, filters=[('county', '==', county_fips)]
        ).set_index('bid')

        # Read NCOneMap parcels
        parcels = read_entities('US-NC_parcel-nconemap-2025', admin3_id, geom=True)
        parcels = parcels[~parcels.drop(columns=['source_deed']).duplicated()].copy()

        # Duplicate parcels are often actual parcels for which the polygon
        # hasn't been carved out of a larger polygon yet - these need to
        # be linked with addresses or through other means. Flag for later.
        parcels['has_duplicate'] = parcels['geo_id'].duplicated(keep=False)
        unique_parcels = parcels[~parcels['geo_id'].duplicated()].copy()

        # After dropping geo_id duplicates, use 'geo_id' as 'parcel_id'
        unique_parcels.index = unique_parcels['geo_id'].rename('parcel_id')

        # Spatially join building centroids with parcels
        buildings_cheer_on_parcels = gpd.sjoin(
            points_from_coords(buildings_cheer),
            unique_parcels[PARCEL_COLUMNS_TO_LINK + ['geometry']],
            how='left',
        )

        # Flag buildings without a parcel
        mask_no_parcel = buildings_cheer_on_parcels['parcel_id'].isnull()
        buildings_cheer_on_parcels.loc[mask_no_parcel, 'issue'] = 'no parcel'

        # Flag buildings linked to duplicated parcels and remove values
        # (for now)
        mask_duplicates = buildings_cheer_on_parcels['parcel_id'].isin(
            parcels[parcels['has_duplicate']]['geo_id'].unique()
        )
        buildings_cheer_on_parcels.loc[mask_duplicates, PARCEL_COLUMNS_TO_LINK] = None
        buildings_cheer_on_parcels.loc[mask_duplicates, 'issue'] = 'duplicate parcel'

        # Flag buildings linked to parcels with multiple buildings
        mask_multiple_buildings = (
            ~mask_duplicates
            & buildings_cheer_on_parcels['parcel_id'].notnull()
            & buildings_cheer_on_parcels['parcel_id'].duplicated(keep=False)
        )
        buildings_cheer_on_parcels.loc[mask_multiple_buildings, 'issue'] = (
            'multi-building value'
        )

        # Insert `admin3_id` behind 'county' so people can get used to it
        buildings_cheer_on_parcels.insert(1, 'admin3_id', admin3_id)

        building_inventory_list += [buildings_cheer_on_parcels]

    print('\nShare inventory')
    building_inventory = pd.concat(building_inventory_list)
    for catcol in CATEGORICAL_COLUMNS:
        building_inventory[catcol] = building_inventory[catcol].astype('category')
    share(building_inventory, out_path, 'share/2026/cheer', delete_original=False)

# Inspect inventory

In [ ]:
print(f'{len(building_inventory):,d} buildings')

## Overview

In [ ]:
building_inventory['issue'].fillna('none (unique link)').value_counts().apply(
    '{0:,d} buildings'.format
)

## Known issues

### Pick a case

In [ ]:
# Issue = 'multi-building value'

# Street view visual/parcel: double-wide mobile home with shed.
# - FEMA wrong, NSI wrong, NCDPS wrong. All claim: 2 SFH.
# - Parcel: $0 improvement value, NCDPS: value n/a
# BID = '87844JGJ+HX4'

# SFH with large garage
# - FEMA wrong: 2 SFH
# - NSI correct, parcel correct: 1 SFH
# BID = '8765P4HJ+JX2'

# SFH with secondary building (garage?)
# - FEMA wrong: 2 SFH
# BID = '8773MJ79+C57'

# Microsoft footprints long gone - perhaps of containers during lot contruction
# BID = '8764754F+34H'

# Commercial mobile home park
# - FEMA: "professional/technical services" $0 land value
# - NSI wrong: SFH with two or three stories
# BID = '87744W96+QG9'

# Blurred
# BID = '87737RHG+QC2'

# building_issue_sample = building_inventory.loc[[BID]]
# building_issue_sample

### Pick random sample of issue

In [ ]:
# ISSUE = 'no parcel'
ISSUE = 'multi-building value'
building_issue_sample = building_inventory.query(f'issue == "{ISSUE}"').sample()
building_issue_sample

In [ ]:
# Infer county `admin3_id`
admin3_id_admin1 = building_issue_sample['county'].values[0]
admin3 = get_admin(CHEER_ADMIN3_IDS, 3).query(
    f'admin3_id_admin1 == "{admin3_id_admin1}"'
)
admin3_id = admin3.index[0]
print(f'{admin3_id}: ' + ' '.join(admin3.iloc[0]))

# Load county datasets
parcels = read_entities('US-NC_parcel-nconemap-2025', admin3_id, geom=True)
buildings_fema = read_entities('US_building-fema-2023', admin3_id, geom=True)
buildings_nsi = read_entities('US_building-nsi-2022', admin3_id, geom=True)
buildings_microsoft = read_entities('US_building-microsoft-v2', admin3_id, geom=True)
buildings_nc = read_entities('US-NC_building-ncdps-2023', admin3_id, geom=True)

In [ ]:
show_building(
    location=building_issue_sample,
    geodatasets={
        'parcels': parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
        'buildings_local': buildings_nc,
    },
)

In [ ]:
# Sample Data
building_issue_sample['state'] = building_issue_sample['admin3_id'].str.slice(3, 5)
data = building_issue_sample[['address', 'city', 'state']].iloc[0]

# Generate the link
sv_link = create_street_view_link(data)
print(sv_link)